# Governance below the data collection: row-level and column-level access in BigQuery

Workbench **data collections** control *who can use a dataset*. They do not mask rows or columns. When
different people may only see **some patients** (one site, one IRB protocol) or **some fields** (no
identifiers), use the data store's own controls underneath the collection. BigQuery offers two:

- **Row access policies**: a filter attached to a table that applies per user or group. People not
  named in any policy see **no rows at all**.
- **Column-level security**: policy tags on sensitive columns; only "Fine-Grained Readers" can select them.

This notebook builds a small **synthetic** table in the workspace's `governance_demo` dataset, adds row
policies for two audiences, and shows the effect live. Nothing here touches Project Baseline data.

**Audiences in this workspace**
- the workspace owner (full access), and
- the Workbench-managed groups that hold access on the dataset; every workspace member is in one of them.

**Requirements:** a BigQuery dataset resource named `governance_demo` in the workspace
(`wb resource create bq-dataset --name=governance_demo`). Row access policies additionally need dataset-owner
rights (see step 2).

**Time:** about 5 minutes. Run it top to bottom once before the session so the table and policies exist.

In [ ]:
import subprocess

import pandas as pd
from google.cloud import bigquery

client = bigquery.Client()

def resolve(resource_name):
    return subprocess.run(["wb", "resolve", "--name", resource_name], capture_output=True, text=True, check=True).stdout.strip()

DATASET = resolve("governance_demo")                     # <project>.governance_demo
TABLE = f"{DATASET}.patients_synthetic"
# Who is "the owner" and who are "the members"? Discover both from the environment:
# the owner is whoever runs this notebook (plus the app's own service account, since notebook queries
# run under it); the members are the Workbench-managed groups that hold access on this dataset.
def _session_identity():
    email = client.query("SELECT SESSION_USER() AS who").to_dataframe().who[0]
    return ("serviceAccount:" if email.endswith("gserviceaccount.com") else "user:") + email

OWNER_GRANTEES = [_session_identity()]
try:
    import requests
    sa = requests.get("http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/default/email",
                      headers={"Metadata-Flavor": "Google"}, timeout=1).text.strip()
    if "@" in sa and f"serviceAccount:{sa}" not in OWNER_GRANTEES:
        OWNER_GRANTEES.append(f"serviceAccount:{sa}")
except Exception:
    pass
MEMBER_GRANTEES = sorted({f"group:{e.entity_id}" for e in client.get_dataset(DATASET).access_entries
                          if e.entity_type == "groupByEmail"})
grantees = lambda items: ", ".join(f"'{g}'" for g in items)
print("owner:", OWNER_GRANTEES)
print("members:", MEMBER_GRANTEES)
print(TABLE)

## 1. A synthetic patient table (24 rows, 3 sites, one obviously sensitive column)

In [ ]:
client.query(f"""
CREATE OR REPLACE TABLE `{TABLE}` AS
WITH base AS (
  SELECT id,
         ['Chicago', 'Peoria', 'Rockford'][OFFSET(MOD(id, 3))] AS site,
         5 + MOD(id * 7, 13) AS age,
         ['Asthma', 'Epilepsy', 'Type 1 diabetes', 'Sickle cell disease'][OFFSET(MOD(id, 4))] AS diagnosis,
         DATE_ADD(DATE '2024-01-01', INTERVAL id * 11 DAY) AS enrollment_date
  FROM UNNEST(GENERATE_ARRAY(1, 24)) AS id)
SELECT FORMAT('P%03d', id) AS patient_id, site, age, diagnosis, enrollment_date,
       FORMAT('MRN-%07d', 1000000 + id * 4817) AS mrn        -- synthetic identifier, to be hidden
FROM base
""").result()
client.query(f"SELECT site, COUNT(*) AS patients FROM `{TABLE}` GROUP BY site ORDER BY site").to_dataframe()

## 2. Row access policies: owner sees everything, workspace members see Chicago only

Creating a row access policy needs `bigquery.rowAccessPolicies.create` on the table, which comes
with **dataset owner** rights. Workbench grants workspace members *dataset editor* and keeps ownership
with its management service, so this cell works only after an org admin or Workbench support grants
`roles/bigquery.dataOwner` on `governance_demo` to the presenter. The cell reports which case you're in.

In [ ]:
from google.api_core.exceptions import Forbidden

POLICIES = {
    "owner_sees_all": (OWNER_GRANTEES, "TRUE"),
    "members_see_chicago": (MEMBER_GRANTEES, "site = 'Chicago'"),
}
policies_active = False
try:
    for name, (who, predicate) in POLICIES.items():
        client.query(f"CREATE OR REPLACE ROW ACCESS POLICY {name} ON `{TABLE}` GRANT TO ({grantees(who)}) FILTER USING ({predicate})").result()
    policies_active = True
    print("Row access policies in place:")
    for p in client.list_row_access_policies(TABLE):
        print(f"  {p.row_access_policy_id}: FILTER USING ({p.filter_predicate})")
except Forbidden as e:
    print("Not permitted from this account:", str(e).split(";")[0][:160])
    print("\nThe SQL an admin would run:")
    for name, (who, predicate) in POLICIES.items():
        print(f"  CREATE ROW ACCESS POLICY {name} ON `{TABLE}`\n    GRANT TO ({grantees(who)})\n    FILTER USING ({predicate});")

## 3. What each audience sees
The same query, run by different people. Policies take a few seconds to apply after creation.

In [ ]:
QUERY = f"SELECT site, COUNT(*) AS patients FROM `{TABLE}` GROUP BY site ORDER BY site"
print("As the workspace owner" + ("" if policies_active else " (no policies active, so everyone sees every row)") + ":")
client.query(QUERY).to_dataframe()

**Live, when policies are active:** ask another workspace member to run the same query in their own app
(JupyterLab, R, or the BigQuery console). They get Chicago only:

```sql
SELECT site, COUNT(*) AS patients
FROM `<project>.governance_demo.patients_synthetic`
GROUP BY site ORDER BY site
```

**No volunteer?** Show it from one seat: drop the owner's policy, and the owner, no longer named in
any policy, sees nothing. Then restore it.

In [ ]:
if policies_active:
    client.query(f"DROP ROW ACCESS POLICY owner_sees_all ON `{TABLE}`").result()
    print("Owner policy removed. Same query, same person:")
    display(client.query(QUERY).to_dataframe())          # expect an empty result
    client.query(f"CREATE OR REPLACE ROW ACCESS POLICY owner_sees_all ON `{TABLE}` GRANT TO ({grantees(OWNER_GRANTEES)}) FILTER USING (TRUE)").result()
    print("Owner policy restored:", len(client.query(QUERY).to_dataframe()), "sites visible again")
else:
    print("Skipped: no row access policies are active on this table.")

## 4. Hiding a column

Row policies don't hide columns. Two options, depending on how strict you need to be:

1. **A view without the column** (shown below). Simple, and right for "analysts should work from the
   de-identified shape". It is not a security boundary while users can still query the base table.
2. **Policy tags (column-level security).** Tag `mrn` with a Data Catalog policy tag; only users granted
   *Fine-Grained Reader* on that tag can select it, everyone else gets an access error even with
   `SELECT *`. This is the enforceable version and is how you'd protect identifiers under a single
   collection. It needs a policy-tag taxonomy in the project, which an org admin sets up once.

In [ ]:
client.query(f"""
CREATE OR REPLACE VIEW `{DATASET}.patients_deidentified` AS
SELECT patient_id, site, age, diagnosis, enrollment_date FROM `{TABLE}`
""").result()
client.query(f"SELECT * FROM `{DATASET}.patients_deidentified` ORDER BY patient_id LIMIT 5").to_dataframe()

## How this maps to common governance questions

| Question | Answer in practice |
|---|---|
| "Collections don't mask rows or columns, so how do we limit access to some patients?" | Row access policies on the table, granted to Workbench groups (which are Google groups), as in step 2 |
| "…or some fields?" | Policy tags for enforcement; de-identified views for convenience (step 4) |
| "One collection per IRB scope?" | Also valid, and simpler to audit. Use it when the scopes are disjoint datasets; use row policies when it's one dataset with per-site or per-protocol slices |

All of it is auditable in Cloud Logging, and none of it requires copying data.

### Cleanup (after the session)
```python
client.delete_table(f"{DATASET}.patients_deidentified", not_found_ok=True)
client.delete_table(TABLE, not_found_ok=True)
```

## Provenance

Generate information about this notebook environment and the packages installed.

In [ ]:
!date

Conda and pip installed packages:

In [ ]:
!conda env export

JupyterLab extensions:

In [ ]:
!jupyter labextension list

Number of cores:

In [ ]:
!grep ^processor /proc/cpuinfo | wc -l

Memory:

In [ ]:
!grep "^MemTotal:" /proc/meminfo

---
Copyright 2026 Verily Life Sciences LLC

Use of this source code is governed by a BSD-style
license that can be found in the LICENSE file or at
https://developers.google.com/open-source/licenses/bsd